# System Design for AI: A Real-Time Voice Agent, Millisecond by Millisecond

The full code of [System Design for AI: A Real-Time Voice Agent, Millisecond by Millisecond](https://residual-stream.lvtan71.workers.dev/posts/voice-agent-system-design/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/voice-agent-system-design) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/voice-agent-system-design.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 0 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "voice-agent-system-design"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/voice-agent-system-design


## `latency_budget.py`

A cascaded voice agent's response time, stage by stage, from assumed figures, and the effect of two levers: semantic turn detection and a speech-to-speech model.

In [2]:
!{sys.executable} latency_budget.py

stage                             ms  cumulative
audio frame + jitter buffer       40          40
network, user -> server           30          70
end-of-turn detection            400         470
speech-to-text, final            100         570
LLM time to first token          250         820
LLM, first clause                150         970
text-to-speech, first audio      120        1090
network, server -> user           30        1120
playback buffer                   40        1160

total 1160 ms; end-of-turn detection alone is 34% of it
  with semantic turn detection (400 -> 150 ms of silence): 910 ms
  with speech-to-speech model (STT + LLM + TTS -> 350 ms): 890 ms


The post printed:

```text
stage                             ms  cumulative
audio frame + jitter buffer       40          40
network, user -> server           30          70
end-of-turn detection            400         470
speech-to-text, final            100         570
LLM time to first token          250         820
LLM, first clause                150         970
text-to-speech, first audio      120        1090
network, server -> user           30        1120
playback buffer                   40        1160

total 1160 ms; end-of-turn detection alone is 34% of it
  with semantic turn detection (400 -> 150 ms of silence): 910 ms
  with speech-to-speech model (STT + LLM + TTS -> 350 ms): 890 ms
```

## `capacity.py`

An idealized simulation of continuous batching for the LLM stage of 1,000 calls on H100s: time to first token and between tokens for 1 to 12 GPUs, with each call's history kept in GPU memory, recomputed every turn, or copied back from CPU memory.

In [3]:
!{sys.executable} capacity.py

1000 calls, a request every 20 s per call; 2,000 + 150 prompt tokens, 60 out
history kept in GPUs  TTFT p50  TTFT p99  ITL p50  ITL p99
            HBM    1   1000 calls' caches don't fit in 80 GB
            HBM    2   500 calls' caches don't fit in 80 GB
            HBM    4   250 calls' caches don't fit in 80 GB


            HBM    6      9 ms     16 ms   5.0 ms   6.2 ms


            HBM    8      8 ms     16 ms   4.9 ms   6.2 ms


            HBM   12      8 ms     14 ms   4.9 ms   6.1 ms


     re-prefill    1   overloaded: requests queue without bound


...

            CPU    2     20 ms     31 ms   5.5 ms   6.8 ms


            CPU    4     19 ms     27 ms   5.1 ms   6.3 ms


            CPU    6     19 ms     27 ms   5.0 ms   6.2 ms


            CPU    8     19 ms     26 ms   4.9 ms   6.2 ms


            CPU   12     18 ms     24 ms   4.9 ms   6.1 ms


The post printed:

```text
1000 calls, a request every 20 s per call; 2,000 + 150 prompt tokens, 60 out
history kept in GPUs  TTFT p50  TTFT p99  ITL p50  ITL p99
            HBM    1   1000 calls' caches don't fit in 80 GB
            HBM    2   500 calls' caches don't fit in 80 GB
            HBM    4   250 calls' caches don't fit in 80 GB
            HBM    6      9 ms     16 ms   5.0 ms   6.2 ms
            HBM    8      8 ms     16 ms   4.9 ms   6.2 ms
            HBM   12      8 ms     14 ms   4.9 ms   6.1 ms
     re-prefill    1   overloaded: requests queue without bound
     re-prefill    2   overloaded: requests queue without bound
     re-prefill    4   8715 ms  22437 ms  20.4 ms 2440.8 ms
     re-prefill    6    194 ms   1321 ms   5.3 ms 261.2 ms
     re-prefill    8    106 ms    554 ms   5.0 ms  87.9 ms
     re-prefill   12     90 ms    349 ms   4.9 ms  87.1 ms
            CPU    1     22 ms     41 ms   6.5 ms  13.3 ms
            CPU    2     20 ms     31 ms   5.5 ms   6.8 ms
            CPU    4     19 ms     27 ms   5.1 ms   6.3 ms
            CPU    6     19 ms     27 ms   5.0 ms   6.2 ms
            CPU    8     19 ms     26 ms   4.9 ms   6.2 ms
            CPU   12     18 ms     24 ms   4.9 ms   6.1 ms
```